# Voronoi diagrams and Delaunay triangulations

A set of sites can define two complementary structures:

- A **Voronoi diagram** partitions the plane into regions of nearest sites.
- A **Delaunay triangulation** connects sites into triangles whose circumcircles are empty of other sites.

They are geometric duals. This lab uses SciPy/Qhull to construct both structures. It does not implement Fortune's sweep-line algorithm or Bowyer–Watson.

In [ ]:
from pathlib import Path
import sys
package_paths = [
    Path.cwd() / "implementations/python/voronoi-delaunay",
    Path.cwd().parent / "implementations/python/voronoi-delaunay",
]
package_path = next((path for path in package_paths if path.is_dir()), None)
if package_path is None:
    raise RuntimeError("Run this notebook from the repository or notebooks directory.")
sys.path.insert(0, str(package_path))

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import voronoi_plot_2d, delaunay_plot_2d
from voronoi_delaunay import build_tessellations

## 1. Construct both structures

The Voronoi region for site p_i is:

$$
V(p_i)=\{x\in\mathbb{R}^2:
\|x-p_i\|\leq\|x-p_j\|\text{ for all }j\}
$$

Every location belongs to the region of its nearest site. Boundaries lie on perpendicular bisectors between sites.

In [ ]:
sites = np.array([
    (0.5, 0.5), (4.5, 0.8), (5.2, 4.2), (2.7, 5.0),
    (0.8, 3.7), (2.4, 2.3), (4.0, 2.7), (1.7, 4.4),
], dtype=float)

result = build_tessellations(sites)
print("Sites:", len(result.points))
print("Voronoi vertices:", len(result.voronoi.vertices))
print("Delaunay triangles:", len(result.delaunay.simplices))

## 2. See the dual structures

The Voronoi view emphasises nearest-site regions. The Delaunay view emphasises connections between sites. A Delaunay edge corresponds to two Voronoi regions sharing a boundary.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
voronoi_plot_2d(result.voronoi, ax=axes[0], show_vertices=True,
                line_colors="tab:blue", line_width=1.8, point_size=18)
axes[0].set_title("Voronoi diagram")
axes[0].set_aspect("equal", adjustable="box")
delaunay_plot_2d(result.delaunay, ax=axes[1])
axes[1].set_title("Delaunay triangulation")
axes[1].set_aspect("equal", adjustable="box")
for ax in axes:
    ax.grid(True, alpha=0.25)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
fig.tight_layout()
plt.show()

## 3. Inspect the Voronoi vertices

A Voronoi vertex is equidistant from three or more sites. Those sites define a Delaunay face around that vertex. Check the distances numerically for this non-degenerate example.

In [ ]:
for index, vertex in enumerate(result.voronoi.vertices):
    distances = np.linalg.norm(result.points - vertex, axis=1)
    nearest = np.argsort(distances)[:3]
    print({
        "vertex": index,
        "coordinate": vertex,
        "three_nearest_site_indices": nearest.tolist(),
        "distances": distances[nearest].round(8).tolist(),
        "spread": float(np.ptp(distances[nearest])),
    })

## 4. Check the Delaunay empty-circumcircle property

For a non-degenerate Delaunay triangle, no other input site lies strictly inside its circumcircle. The next cell calculates each triangle's circumcentre and checks the remaining sites.

In [ ]:
def circumcircle(triangle):
    a, b, c = triangle
    matrix = 2 * np.array([b - a, c - a])
    rhs = np.array([b @ b - a @ a, c @ c - a @ a])
    centre = np.linalg.solve(matrix, rhs)
    radius_squared = np.sum((centre - a) ** 2)
    return centre, radius_squared

for simplex in result.delaunay.simplices:
    triangle = result.points[simplex]
    centre, radius2 = circumcircle(triangle)
    distances2 = np.sum((result.points - centre) ** 2, axis=1)
    other_sites = np.ones(len(result.points), dtype=bool)
    other_sites[simplex] = False
    assert np.all(distances2[other_sites] >= radius2 - 1e-9)
print("Every returned triangle passed the empty-circumcircle check.")

## 5. Understand the Fortune sweep at a conceptual level

Fortune's algorithm constructs the Voronoi diagram with a plane sweep. It maintains a beach line made of parabolic arcs and processes two event families:

- **Site events:** a new site is reached and its arc is inserted.
- **Circle events:** a beach-line arc disappears, identifying a Voronoi vertex.

A priority queue orders events; a balanced search structure maintains the beach line. The original algorithm achieves $O(n\log n)$ time and $O(n)$ space for point sites under its stated model. This notebook does not implement that sweep.

## 6. Degeneracies are part of the model

Duplicate sites and collinear inputs are rejected by the wrapper. Cocircular sites can admit more than one valid Delaunay diagonal; do not assert one arbitrary diagonal unless the implementation's tie policy is specified. Voronoi cells may be unbounded, and a plotting window does not become part of the mathematical diagram.

In [ ]:
for label, invalid_sites in {
    "duplicate": [(0, 0), (1, 0), (0, 1), (0, 0)],
    "collinear": [(0, 0), (1, 1), (2, 2)],
}.items():
    try:
        build_tessellations(invalid_sites)
    except ValueError as error:
        print(label, "->", error)

## Your turn

1. Add a new site near the centre. Which Voronoi regions change?
2. Move a site outside the current point cloud. Which Delaunay triangles are replaced?
3. Create four cocircular sites. Why might either diagonal be valid?
4. Add a duplicate point. Why should the input contract reject it?
5. Compare the library-backed result with the conceptual Fortune sweep. Which structures would you need to implement to build Fortune's algorithm yourself?

**Implementation:** [SciPy/Qhull wrapper](../implementations/python/voronoi-delaunay/README.md) and [intern walkthrough](../implementations/python/voronoi-delaunay/implementation-walkthrough.md).